In [8]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import automl, Input

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected:", ml_client.workspaces.get("ml-learning-workspace").name)

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected: ml-learning-workspace


In [2]:
import pandas as pd
from sklearn.datasets import load_iris
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
import os

# Create iris dataframe
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

# Save locally
os.makedirs("data", exist_ok=True)
df.to_csv("data/iris.csv", index=False)
print("Dataset saved, shape:", df.shape)
print(df.head())

Dataset saved, shape: (150, 5)
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


In [3]:
# Upload and register dataset in Azure ML
data_asset = Data(
    path="data/iris.csv",
    type=AssetTypes.URI_FILE,
    name="iris-dataset",
    description="Iris classification dataset for AutoML"
)

registered_data = ml_client.data.create_or_update(data_asset)
print(f"Dataset registered: {registered_data.name}, version: {registered_data.version}")

Uploading iris.csv (< 1 MB): 100%|██████████| 2.78k/2.78k [00:00<00:00, 216kB/s]




Dataset registered: iris-dataset, version: 1


In [4]:
from azure.ai.ml import automl, Input
from azure.ai.ml.constants import AssetTypes

# Create AutoML classification job
automl_job = automl.classification(
    compute="ml-compute-md",
    experiment_name="iris-automl",
    training_data=Input(
        type=AssetTypes.URI_FILE,
        path=f"azureml:iris-dataset:1"
    ),
    target_column_name="target",
    primary_metric="accuracy",
    n_cross_validations=5,
)

# Set limits to keep it cheap and fast
automl_job.set_limits(
    timeout_minutes=20,
    trial_timeout_minutes=5,
    max_trials=5
)

# Submit job
returned_job = ml_client.jobs.create_or_update(automl_job)
print(f"Job submitted: {returned_job.name}")
print(f"Job status: {returned_job.status}")
print(f"View in studio: {returned_job.studio_url}")

Job submitted: modest_date_6nhwz4bgyz
Job status: NotStarted
View in studio: https://ml.azure.com/runs/modest_date_6nhwz4bgyz?wsid=/subscriptions/749d055e-0977-4255-acd4-54c54916bff8/resourcegroups/ml-learning-rg/workspaces/ml-learning-workspace&tid=5fb38fb2-979b-407e-8873-dec137663a6a


In [5]:
import os
import pandas as pd
from sklearn.datasets import load_iris

# Create MLTable format
os.makedirs("data/iris-mltable", exist_ok=True)

# Save CSV inside the mltable folder
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df.to_csv("data/iris-mltable/iris.csv", index=False)

# Create MLTable yaml file
mltable_yaml = """paths:
  - file: ./iris.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
"""

with open("data/iris-mltable/MLTable", "w") as f:
    f.write(mltable_yaml)

print("MLTable created successfully")

MLTable created successfully


In [6]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import automl, Input

# Register as MLTable
data_asset = Data(
    path="data/iris-mltable",
    type=AssetTypes.MLTABLE,
    name="iris-mltable",
    description="Iris dataset in MLTable format"
)

registered_data = ml_client.data.create_or_update(data_asset)
print(f"Dataset registered: {registered_data.name}, version: {registered_data.version}")

# Resubmit AutoML job
automl_job = automl.classification(
    compute="ml-compute-md",
    experiment_name="iris-automl",
    training_data=Input(
        type=AssetTypes.MLTABLE,
        path=f"azureml:iris-mltable:1"
    ),
    target_column_name="target",
    primary_metric="accuracy",
    n_cross_validations=5,
)

automl_job.set_limits(
    timeout_minutes=20,
    trial_timeout_minutes=5,
    max_trials=5
)

returned_job = ml_client.jobs.create_or_update(automl_job)
print(f"Job submitted: {returned_job.name}")
print(f"View in studio: {returned_job.studio_url}")

Uploading iris-mltable (0.0 MBs): 100%|██████████| 2921/2921 [00:00<00:00, 38472.24it/s]




Dataset registered: iris-mltable, version: 1
Job submitted: crimson_bag_2p392qxt8b
View in studio: https://ml.azure.com/runs/crimson_bag_2p392qxt8b?wsid=/subscriptions/749d055e-0977-4255-acd4-54c54916bff8/resourcegroups/ml-learning-rg/workspaces/ml-learning-workspace&tid=5fb38fb2-979b-407e-8873-dec137663a6a
